# What Is a Qubit, in Words

Intuition first. Math after.

**Objectives:**
- Explain a qubit in plain English using the spinning-coin metaphor
- Map that metaphor onto a unit vector in $\mathbb{C}^2$
- Compute measurement probabilities of `|0>`, `|1>`, `|+>`, `|->` by hand
- Be ready to read `|psi> = alpha|0> + beta|1>` without flinching

**Reference:** See [`../GUIDE.md`](../GUIDE.md).

<!-- browser-runnable -->


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)

## 1. The spinning-coin metaphor

- A **classical bit** is a coin lying flat on the table. Heads or tails. That's it.
- A **qubit** is a coin you have just **spun**. While it spins, it is leaning   in some direction. The instant you stop the spin (measure), it falls to   heads or tails — but *which* one is probabilistic, and the probability   depends on the direction it was leaning.

So a qubit has more structure than a bit: it carries a *direction*, not just a value. The direction is what makes superposition and interference possible. The value (0 or 1) only appears at measurement.

The rest of this notebook makes "direction" precise.

## 2. The math version: a unit 2-vector

A qubit state is a 2-vector

$$|\psi\rangle = \begin{pmatrix} \alpha \\ \beta \end{pmatrix}$$

with complex entries `alpha`, `beta` satisfying $|\alpha|^2 + |\beta|^2 = 1$.

The two **computational basis states** are

$$|0\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix}, \qquad |1\rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix}.$$

Every other state is some combination of those two.

In [ ]:
# Build the four canonical states.
zero  = np.array([1, 0], dtype=complex)
one   = np.array([0, 1], dtype=complex)
plus  = (zero + one) / np.sqrt(2)        # |+> = (|0> + |1>) / sqrt(2)
minus = (zero - one) / np.sqrt(2)        # |-> = (|0> - |1>) / sqrt(2)

for name, state in [('|0>', zero), ('|1>', one), ('|+>', plus), ('|->', minus)]:
    print(f'{name}: {state}    norm={np.linalg.norm(state):.3f}')

## 3. Measurement, formally

If $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ then measuring in the computational basis gives outcome **0** with probability $|\alpha|^2$ and outcome **1** with probability $|\beta|^2$.

That is the **Born rule**, and it is the only mysterious thing in quantum mechanics. Once you accept it, the rest is linear algebra.

In [ ]:
def measurement_probs(psi):
    return np.abs(psi) ** 2

for name, state in [('|0>', zero), ('|1>', one), ('|+>', plus), ('|->', minus)]:
    p = measurement_probs(state)
    print(f'{name}: P(0) = {p[0]:.3f},  P(1) = {p[1]:.3f}')

**Read this slowly:** `|+>` and `|->` give *identical* measurement probabilities (50/50). They are different states — but you cannot tell them apart with a single computational-basis measurement. To distinguish them you have to *do* something to the state first (e.g. apply a Hadamard) and then measure. That is what gates are for.

## 4. Simulate measurement

Combining notebook 3 with what we just learned: "running a quantum circuit with N shots" is *literally* sampling N times from the distribution `|psi|^2`.

In [ ]:
def simulate_measurements(psi, n_shots=1000, seed=0):
    rng = np.random.default_rng(seed)
    probs = measurement_probs(psi)
    return rng.choice(len(psi), size=n_shots, p=probs)

shots = simulate_measurements(plus, n_shots=2000)
p0 = np.mean(shots == 0); p1 = np.mean(shots == 1)
print(f'|+> measured 2000 times: P(0) ~ {p0:.3f}, P(1) ~ {p1:.3f}')

In [ ]:
# Plot all four canonical states for comparison.
states = [('|0>', zero), ('|1>', one), ('|+>', plus), ('|->', minus)]
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for ax, (name, psi) in zip(axes, states):
    p = measurement_probs(psi)
    ax.bar(['0', '1'], p, color=['#ff9900', '#146eb4'])
    ax.set_title(name)
    ax.set_ylim(0, 1)
axes[0].set_ylabel('Probability')
plt.suptitle('Measurement distributions of the four canonical states')
plt.tight_layout()
plt.show()

## 5. A non-canonical state

Let's build a custom state and watch the probabilities behave.

In [ ]:
# 30% chance of 0, 70% chance of 1
psi = np.array([np.sqrt(0.3), np.sqrt(0.7)], dtype=complex)
print('state:', psi)
print('norm:', np.linalg.norm(psi))           # 1
print('measurement probs:', measurement_probs(psi))

shots = simulate_measurements(psi, n_shots=10_000, seed=1)
print(f'empirical: P(0) ~ {np.mean(shots == 0):.3f}, P(1) ~ {np.mean(shots == 1):.3f}')

## 6. Exercises

Three exercises to make the Born rule automatic. Each has tiered hints --
expand only what you need -- and a check cell that tells you when you have
it. Worked solutions wait at the bottom of the notebook, but give the
checks an honest try first.

### Exercise 1 — Amplitudes to probabilities

Build the state $|\psi\rangle = \tfrac{1}{2}|0\rangle + \tfrac{\sqrt{3}}{2}|1\rangle$ and work out what a computational-basis
measurement gives.

Define `psi_ex1`: the state as a length-2 complex NumPy array, and
`probs_ex1`: its two measurement probabilities `[P(0), P(1)]`.

<details><summary>Hint 1 — nudge</summary>

The coefficients in front of $|0\rangle$ and $|1\rangle$ are amplitudes,
not probabilities. The Born rule from Section 3 says how to get from one
to the other.

</details>
<details><summary>Hint 2 — approach</summary>

Build the vector with `np.array([...], dtype=complex)` using the two
amplitudes from the prompt, then reuse the notebook's `measurement_probs`
helper (or take `np.abs(...) ** 2` yourself).

</details>

In [ ]:
# Exercise 1: Build psi = (1/2)|0> + (sqrt(3)/2)|1> and find P(0), P(1).
# Define: psi_ex1 -- the state vector; probs_ex1 -- its [P(0), P(1)].

# TODO: your code here

In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    assert np.asarray(psi_ex1).shape == (2,), "a single-qubit state is a 2-vector"
    assert np.isclose(np.linalg.norm(psi_ex1), 1.0), "a valid state has norm 1"
    assert np.isclose(np.sum(probs_ex1), 1.0), (
        "measurement probabilities must sum to 1"
    )
    assert np.allclose(probs_ex1, np.abs(np.asarray(psi_ex1)) ** 2), (
        "probs_ex1 should be the Born-rule probabilities of your psi_ex1"
    )
    assert np.isclose(probs_ex1[1], 3 * probs_ex1[0]), (
        "with these amplitudes, outcome 1 should be exactly three times "
        "as likely as outcome 0"
    )

### Exercise 2 — An imaginary amplitude

A qubit is prepared in $|\psi\rangle = \tfrac{1}{\sqrt{2}}|0\rangle + \tfrac{i}{\sqrt{2}}|1\rangle$ -- note the imaginary amplitude on
$|1\rangle$. What are the measurement probabilities?

Define `psi_ex2`: that state as a complex NumPy array, and `probs_ex2`:
its measurement probabilities `[P(0), P(1)]`.

<details><summary>Hint 1 — nudge</summary>

The Born rule squares the *modulus* of each amplitude, and the modulus is
the complex magnitude -- the direction of an amplitude in the complex
plane drops out entirely.

</details>
<details><summary>Hint 2 — approach</summary>

In Python the imaginary unit is `1j`. Build the array with the two
amplitudes from the prompt and push it through `measurement_probs` --
`np.abs` already takes the complex modulus, so no special handling is
needed.

</details>

In [ ]:
# Exercise 2: Measurement probabilities of psi = [1/sqrt(2), 1j/sqrt(2)].
# Define: psi_ex2 -- the state vector; probs_ex2 -- its [P(0), P(1)].

# TODO: your code here

In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    _psi2 = np.asarray(psi_ex2)
    assert _psi2.shape == (2,), "a single-qubit state is a 2-vector"
    assert abs(_psi2[1].imag) > 0.1, (
        "keep the amplitude on |1> imaginary, exactly as the prompt writes it"
    )
    assert np.isclose(np.linalg.norm(_psi2), 1.0), "a valid state has norm 1"
    assert np.isclose(np.sum(probs_ex2), 1.0), (
        "measurement probabilities must sum to 1"
    )
    assert np.all(np.asarray(probs_ex2) >= 0), "probabilities are never negative"
    assert np.isclose(probs_ex2[0], probs_ex2[1]), (
        "the imaginary unit is a phase -- it should not tilt the odds either way"
    )

### Exercise 3 — Two states, one distribution

`|+>` and `|->` are genuinely different states, yet Section 3 claims a
computational-basis measurement cannot tell them apart. Confirm that
numerically, then say *why* in one sentence (out loud, or in a comment).

Define `probs_plus3`: the measurement probabilities of `|+>`, and
`probs_minus3`: the measurement probabilities of `|->`.

<details><summary>Hint 1 — nudge</summary>

Where does the difference between $|+\rangle$ and $|-\rangle$ live -- in
the size of the amplitudes, or in their sign? Which of those does the
Born rule see?

</details>
<details><summary>Hint 2 — approach</summary>

The states `plus` and `minus` were built back in Section 2. Push each
through `measurement_probs` (or `np.abs(...) ** 2`) and compare the two
arrays entry by entry.

</details>

In [ ]:
# Exercise 3: Show that |+> and |-> measure identically, then explain why.
# Define: probs_plus3 -- measurement probabilities of |+>;
#         probs_minus3 -- measurement probabilities of |->.

# TODO: your code here

In [ ]:
# Check Exercise 3 -- run after your attempt.
from lib.grading import check

with check("Exercise 3"):
    assert np.asarray(probs_plus3).shape == (2,), (
        "probs_plus3 should hold [P(0), P(1)]"
    )
    assert np.asarray(probs_minus3).shape == (2,), (
        "probs_minus3 should hold [P(0), P(1)]"
    )
    assert np.isclose(np.sum(probs_plus3), 1.0) and np.isclose(
        np.sum(probs_minus3), 1.0
    ), "each set of measurement probabilities must sum to 1"
    assert np.allclose(probs_plus3, probs_minus3), (
        "the two distributions should be indistinguishable -- that is the point"
    )
    assert np.isclose(probs_plus3[0], probs_plus3[1]), (
        "each state should look like a fair coin in the computational basis"
    )

### Solutions

In [ ]:
# --- Exercise 1 ---
psi_ex1 = np.array([1/2, np.sqrt(3)/2], dtype=complex)
probs_ex1 = measurement_probs(psi_ex1)
print(probs_ex1)    # [0.25, 0.75]

# --- Exercise 2 ---
psi_ex2 = np.array([1, 1j]) / np.sqrt(2)
probs_ex2 = measurement_probs(psi_ex2)
print(probs_ex2)    # [0.5, 0.5] — imaginary part doesn't change |.|^2

*Answer (Exercise 3): Because the Born rule depends on $|\alpha|^2$ and $|\beta|^2$ only, the **sign** of $\beta$ does not show up in a computational-basis measurement. To detect the sign difference you have to apply a gate (specifically, a Hadamard) before measuring.*

In [ ]:
# --- Exercise 3 (numerical confirmation) ---
probs_plus3 = measurement_probs(plus)
probs_minus3 = measurement_probs(minus)
print('|+|^2 =', probs_plus3)     # [0.5, 0.5]
print('|-|^2 =', probs_minus3)    # [0.5, 0.5] — identical to |+|^2

**You finished notebook 4.** Move on to [`05-dirac-notation-decoded.ipynb`](05-dirac-notation-decoded.ipynb).